# Amazon Bedrock AgentCore Gateway - 시맨틱 검색 튜토리얼

### 튜토리얼 세부 정보


| 정보                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 대화형                                                                            |
| 에이전트 유형       | 단일                                                                              |
| AgentCore 서비스    | AgentCore Gateway, AgentCore Identity                                            |
| 에이전틱 프레임워크 | Strands Agents                                                                   |
| LLM 모델            | Anthropic Claude Haiku 4.5                                                        |
| 튜토리얼 구성 요소  | Strands Agent에서 AWS Lambda 기반 AgentCore Gateway 생성 및 사용                 |
| 튜토리얼 분야       | 범분야                                                                            |
| 예제 난이도         | 쉬움                                                                              |
| 사용 SDK            | Amazon BedrockAgentCore Python SDK 및 boto3  

### 튜토리얼 아키텍처
Amazon Bedrock AgentCore Gateway는 에이전트와 에이전트가 상호 작용하는 데 필요한 도구 및 리소스 사이에 통합 연결을 제공합니다. Gateway는 이 연결 계층에서 여러 역할을 수행합니다.

1. **보안 관리자**: Gateway는 유효한 사용자와 에이전트만 도구 및 리소스에 접근하도록 OAuth 권한 부여를 관리합니다.
2. **변환기**: Gateway는 Model Context Protocol(MCP)과 같은 널리 사용되는 프로토콜을 이용한 에이전트 요청을 API 요청과 Lambda 호출로 변환합니다. 따라서 개발자가 서버를 호스팅하거나 프로토콜 통합, 버전 지원, 버전 패치 등을 관리할 필요가 없습니다.
3. **구성 도구**: Gateway를 사용하면 개발자가 여러 API, 함수, 도구를 에이전트가 사용할 수 있는 단일 MCP 엔드포인트로 원활하게 결합할 수 있습니다.
4. **키 체인**: Gateway는 각 도구에 적합한 자격 증명을 주입하여, 에이전트가 서로 다른 자격 증명 집합이 필요한 도구를 원활하게 활용하도록 합니다. 
5. **검색 도구**: Gateway는 에이전트가 모든 도구를 검색하여 주어진 컨텍스트나 질문에 가장 적합한 도구만 찾을 수 있게 합니다. 이를 통해 에이전트는 소수의 도구가 아니라 수천 개의 도구를 활용할 수 있습니다. 또한 에이전트의 LLM 프롬프트에 제공해야 하는 도구 집합을 최소화하여 지연 시간과 비용을 줄입니다. 
6. **인프라 관리자**: Gateway는 완전한 서버리스 서비스이며 관찰성과 감사 기능이 기본 제공되므로, 개발자가 에이전트와 도구를 통합하기 위한 추가 인프라를 관리할 필요가 줄어듭니다. 

![작동 방식](images/gw-arch-overview.png)

### 튜토리얼 주요 기능

* AWS Lambda 기반 대상을 사용하여 Amazon Bedrock AgentCore Gateway 생성
* AgentCore Gateway 시맨틱 검색 사용 
* Strands Agents를 사용하여 AgentCore Gateway 검색이 지연 시간을 개선하는 방식 확인

## 사전 요구 사항

이 튜토리얼을 실행하려면 다음 항목이 필요합니다.
* Python 3.10+
* AWS 자격 증명
* Amazon Bedrock AgentCore SDK
* Strands Agents

## AgentCore Gateway로 수많은 도구를 보유한 MCP 서버의 문제 해결
일반적인 엔터프라이즈 환경에서 에이전트 개발자는 수백 개에서 수천 개에 이르는
MCP 도구를 보유한 MCP 서버를 접하게 됩니다. 이렇게 많은 도구는 AI 에이전트의 도구 선택 정확도를 낮추고, 
비용을 늘리며, 과도한 도구 메타데이터로 인한 토큰 사용량 증가로 지연 시간을 높이는 문제를 일으킵니다.
이 문제는 에이전트를 서드 파티 서비스(예: Zendesk, Salesforce,
Slack, JIRA 등) 또는 기존 엔터프라이즈 REST 서비스에 연결할 때 발생할 수 있습니다. 
AgentCore Gateway는 도구 전반에 걸친 시맨틱 검색을 기본 제공하여, 
에이전트에 필요한 도구를 계속 제공하면서도 에이전트의 지연 시간, 비용, 정확도를 개선합니다. 
사용 사례, LLM 모델, 에이전트 프레임워크에 따라 일반적인 MCP Server의 수백 개 도구 전체를 제공하는 대신
관련 도구에 에이전트를 집중시켜 지연 시간을 최대 3배까지 개선할 수 있습니다.

![작동 방식](images/gateway_tool_search.png)

## 이 Notebook에서 학습할 내용
이 Notebook에서는 AgentCore Gateway 검색 튜토리얼을 제공합니다. 이 단계별 튜토리얼을 마치면 다음 내용을
이해할 수 있습니다.

- AgentCore Gateway의 기본 제공 검색 도구를 사용하여 관련 도구를 빠르게 찾는 방법 
- 도구 검색 결과를 Strands Agents에 통합하여 지연 시간을 개선하고 비용을 줄이는 방법

## Notebook 구성 개요
Notebook은 다음 섹션으로 구성됩니다.

1. AgentCore Gateway 검색의 기본 원리 이해
2. Notebook 환경 준비
3. 수백 개의 도구를 보유한 Gateway 설정
4. Gateway에서 도구 검색
5. 도구가 많은 MCP 서버에서 Strands Agents 사용
6. Strands Agent에 도구 검색 결과 추가
7. 도구 검색을 통한 3배의 지연 시간 개선 확인

# AgentCore Gateway 검색의 기본 원리 이해

AgentCore Gateway를 생성할 때 검색을 활성화하도록 지정할 수 있습니다.
검색이 활성화된 Gateway에서는 다음 세 가지 작업이 이루어집니다.

1. **벡터 스토어가 생성됩니다**. Gateway 서비스는 새 Gateway를 위한 서버리스 완전 관리형 벡터 스토어를 자동으로 생성합니다. 이를 통해 Gateway 도구 전체를 대상으로 시맨틱 검색을 수행할 수 있습니다. 
3. **벡터 스토어가 채워집니다**. Gateway에 Gateway Target을 추가하면 서비스가 내부적으로 임베딩을 자동 사용하여 새 Target의 도구를 기반으로 벡터 스토어를 채웁니다. 도구 메타데이터는 도구의 JSON 정의 또는 REST 서비스 대상의 OpenAPI Schema 사양에서 가져옵니다.
2. **검색 도구(MCP 기반)가 제공됩니다**. Gateway에는 사용자가 정의한 모든 도구(AWS Lambda 대상 또는 REST 서비스의 도구) 외에도 시맨틱 검색을 제공하는 MCP 도구 하나가 추가됩니다. 이 도구의 이름은 `x-amz-bedrock-agentcore-search`입니다. 이 접두사는 사용자 정의 도구와 이름이 충돌하지 않도록 합니다. 향후 이와 같은 도구가 더 추가될 수도 있습니다. 검색 도구에는 `query`라는 단일 인수가 있습니다. 검색 도구가 호출되면 Gateway 서비스는 해당 쿼리를 사용하여 시맨틱 검색을 수행하고, 사용 가능한 도구 메타데이터(이름, 설명, 입력 및 출력 스키마)와 일치시킨 다음, 관련성이 높은 순서대로 가장 적합한 도구를 반환합니다.

# Notebook 환경 준비

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

필요한 모든 Python 라이브러리를 가져오고 환경 변수를 로드합니다.

In [ ]:
from strands import Agent
from strands.models import BedrockModel
from strands.handlers import null_callback_handler

from strands.tools.mcp.mcp_client import MCPClient, MCPAgentTool

from mcp.client.streamable_http import streamablehttp_client
from mcp.types import Tool as MCPTool

import logging
import time
import json
import boto3
import requests
import utils

GATEWAY_NAME = "gateway-search-tutorial"

로거를 설정합니다.

In [ ]:
# 루트 strands 로거 구성
logging.getLogger("strands").setLevel(logging.ERROR)  # INFO) #DEBUG) #

# 로그를 확인할 수 있도록 핸들러 추가
logging.basicConfig(format="%(levelname)s | %(name)s | %(message)s", handlers=[logging.StreamHandler()])

boto3 버전을 확인합니다.

In [ ]:
boto3.__version__

AgentCore 컨트롤 플레인 API용 boto3 클라이언트를 가져옵니다.

In [ ]:
session = boto3.Session()
agentcore_client = session.client(
    "bedrock-agentcore-control",
)

# 수백 개의 도구를 보유한 Gateway 설정

AgentCore Gateway는 선별된 기존 API 집합을 에이전트용 MCP 도구로 노출하는
안전하고 확장 가능한 방법을 제공합니다. 프로덕션 환경에서는 CloudFormation, CDK 또는 Terraform과 같은 도구로
코드형 인프라를 사용하여 Gateway 리소스를 생성합니다. 이 튜토리얼에서는
리소스와 API를 더 효과적으로 이해할 수 있도록 boto3 컨트롤 플레인 API를 직접 사용합니다.
이를 통해 자체 Gateway 구축 및 사용을 더 쉽게 시작하고, 더욱 강력하고
안전한 에이전트를 만들 수 있습니다.

Gateway 설정 단계의 개요는 다음과 같습니다.

1. 인바운드(에이전트가 Gateway 호출) 및 아웃바운드(Gateway가 도구 호출) 보안에 사용할 자격 증명 공급자와 ID 제공업체를 정의합니다.
2. `create_gateway`를 사용하여 Gateway를 생성합니다.
3. `create_gateway_target`을 사용하여 Gateway Target을 추가하고, AWS Lambda 또는 기존 RESTful 서비스에 구현된 MCP 도구를 노출합니다.

이 튜토리얼에서는 Amazon Cognito를 ID 제공업체(IdP)로, AWS Lambda 함수를 대상으로, AWS IAM을 아웃바운드 인증에 사용합니다. 이 튜토리얼에서 설명하는 개념은 다른 IdP나 다른 대상 유형을 사용할 때도 동일하게 적용됩니다.

### Amazon Cognito 리소스 생성

이 튜토리얼에서는 다음 리소스를 이미 생성하고 해당 환경 변수를 설정했다고 가정합니다.

- AWS Lambda 실행용 IAM 역할(`gateway_lambda_iam_role`)
- 간단한 수학 도구용 AWS Lambda 함수(`calc_lambda_arn`)
- 레스토랑 예약 도구용 AWS Lambda 함수(`restaurant_lambda_arn`)
- 클라이언트 ID(`cognito_client_id`)와 검색 URL(`cognito_discovery_url`)을 제공하는 Amazon Cognito 사용자 풀

레스토랑 API의 JSON 도구 메타데이터를 살펴보겠습니다. 기존 REST 서비스와 통합하는 경우에는 OpenAPI Schema를 사용하여 API 사양을 제공한다는 점에 유의하세요.

In [ ]:
with open("./restaurant/restaurant-api.json") as f:
    data = json.load(f)
print(json.dumps(data, indent=4))

다음은 간단한 계산기 API입니다.

In [ ]:
with open("./calc/calc-api.json") as f:
    data = json.load(f)[0:3]
print(json.dumps(data, indent=4))

다음은 계산기 도구의 AWS Lambda 함수 구현입니다.

In [ ]:
from IPython.display import display, Code

with open("./calc/lambda_function_code.py", "r") as f:
    code_content = f.read()
display(Code(code_content, language="python"))

In [ ]:
with open("./restaurant/lambda_function_code.py", "r") as f:
    code_content = f.read()
display(Code(code_content, language="python"))

In [ ]:
#### MCP 도구로 변환할 샘플 AWS Lambda 함수 생성
calc_lambda_resp = utils.create_gateway_lambda(
    "calc/lambda_function_code.zip", lambda_function_name="calc_lambda_gateway"
)

if calc_lambda_resp is not None:
    if calc_lambda_resp["exit_code"] == 0:
        print(
            "Lambda function created with ARN: ",
            calc_lambda_resp["lambda_function_arn"],
        )
    else:
        print(
            "Lambda function creation failed with message: ",
            calc_lambda_resp["lambda_function_arn"],
        )

In [ ]:
calc_lambda_resp["lambda_function_arn"]

In [ ]:
#### MCP 도구로 변환할 샘플 AWS Lambda 함수 생성
restaurant_lambda_resp = utils.create_gateway_lambda(
    "restaurant/lambda_function_code.zip",
    lambda_function_name="restaurant_lambda_gateway",
)

if restaurant_lambda_resp is not None:
    if restaurant_lambda_resp["exit_code"] == 0:
        print(
            "Lambda function created with ARN: ",
            restaurant_lambda_resp["lambda_function_arn"],
        )
    else:
        print(
            "Lambda function creation failed with message: ",
            restaurant_lambda_resp["lambda_function_arn"],
        )

In [ ]:
restaurant_lambda_resp["lambda_function_arn"]

In [ ]:
cognito_response = utils.setup_cognito_user_pool()

In [ ]:
bearer_token = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)

In [ ]:
gateway_role_arn = utils.create_gateway_iam_role(
    lambda_arns=[
        calc_lambda_resp["lambda_function_arn"],
        restaurant_lambda_resp["lambda_function_arn"],
    ]
)

#### 컨트롤 플레인 API 사용을 위한 몇 가지 헬퍼 함수 생성

In [ ]:
def read_apispec(json_file_path):
    try:
        # JSON 파일을 읽고 내용을 문자열로 반환
        with open(json_file_path, "r") as file:
            # JSON을 Python 객체로 파싱
            api_spec = json.load(file)
            return api_spec

    except FileNotFoundError:
        return f"Error: File {json_file_path} not found"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"


def list_gateways():
    response = agentcore_client.list_gateways()
    print(json.dumps(response, indent=2, default=str))
    return response

#### Gateway 생성 헬퍼 함수
다음은 이름과 설명을 받아 AgentCore Gateway를 생성하는 헬퍼 함수입니다.
Amazon Cognito를 IdP로 사용하며, 허용된 클라이언트 ID와 검색 URL은
이미 정의된 환경 변수에서 가져옵니다.
또한 생성된 Gateway에서 시맨틱 검색을 기본적으로 활성화하고
미리 정의된 IAM 역할을 사용합니다.

In [ ]:
def create_gateway(gateway_name, gateway_desc):
    # Gateway의 인바운드 OAuth에 Cognito 사용
    auth_config = {
        "customJWTAuthorizer": {
            "allowedClients": [cognito_response["client_id"]],
            "discoveryUrl": cognito_response["discovery_url"],
        }
    }
    # 도구 시맨틱 검색 활성화
    search_config = {"mcp": {"searchType": "SEMANTIC", "supportedVersions": ["2025-03-26"]}}
    # Gateway 생성
    response = agentcore_client.create_gateway(
        name=gateway_name,
        roleArn=gateway_role_arn,
        authorizerType="CUSTOM_JWT",
        description=gateway_desc,
        protocolType="MCP",
        authorizerConfiguration=auth_config,
        protocolConfiguration=search_config,
    )
    # print(json.dumps(response, indent=2, default=str))
    return response["gatewayId"]

#### Gateway Target 생성 헬퍼 함수
이 함수는 기존 Gateway에 새 AWS Lambda 대상을 생성합니다.
Gateway ID, 새 대상의 이름과 설명, 기존 AWS Lambda 함수의 ARN,
그리고 Gateway에서 노출할 도구의 인터페이스를 설명하는 JSON 스키마만
제공하면 됩니다.

In [ ]:
def create_gatewaytarget(gateway_id, target_name, target_descr, lambda_arn, api_spec):
    # Gateway에 Lambda 대상 추가
    response = agentcore_client.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name=target_name,
        description=target_descr,
        targetConfiguration={
            "mcp": {
                "lambda": {
                    "lambdaArn": lambda_arn,
                    "toolSchema": {"inlinePayload": api_spec},
                }
            }
        },
        # IAM을 자격 증명 공급자로 사용
        credentialProviderConfigurations=[{"credentialProviderType": "GATEWAY_IAM_ROLE"}],
    )
    return response["targetId"]

### 첫 번째 AgentCore Gateway 생성
첫 번째 Gateway를 설정하기 전에 Gateway가 MCP 도구 사용을 위한 인바운드 요청과
Gateway에서 도구 및 리소스로 향하는 아웃바운드 접근 모두에
보안을 제공하는 방식을 간단히 살펴보겠습니다.

![작동 방식](images/gateway_secure_access.png)

이제 이름과 설명을 지정하여 이 튜토리얼에서 사용할 Gateway를 생성하겠습니다.

In [ ]:
print(f"Create gateway with name: {GATEWAY_NAME}")
gatewayId = create_gateway(gateway_name=GATEWAY_NAME, gateway_desc="AgentCore Gateway Tutorial")
print(f"Gateway created with id: {gatewayId}.")

### AgentCore Gateway Target 추가
이 튜토리얼에서는 간단한 수학 계산을 수행하는 함수와 레스토랑 예약 생성을 시뮬레이션하는 함수로 구성된
두 개의 Lambda 함수가 이미 설치되어 있다고 가정합니다. 각 함수에 대해
Gateway Target을 추가하겠습니다.

이 대상을 추가한 다음에는 MCP 도구 수를 늘리기 위한 대상을 추가하여
AgentCore Gateway 검색의 강력한 기능을 시연하겠습니다.

Gateway를 생성했으므로 이제 Lambda 함수를 통해 레스토랑을 예약하는
대상을 추가하겠습니다.

In [ ]:
restaurant_api_spec = read_apispec("./restaurant/restaurant-api.json")
restaurant_lambda_arn = restaurant_lambda_resp["lambda_function_arn"]
print(f"Restaurant Lambda ARN: {restaurant_lambda_arn}")

restaurantTargetId = create_gatewaytarget(
    gateway_id=gatewayId,
    lambda_arn=restaurant_lambda_arn,
    target_name="FoodTools",
    target_descr="Restaurant Tools",
    api_spec=restaurant_api_spec,
)
print(f"RestaurantTarget created with id: {restaurantTargetId} on gateway: {gatewayId}")

여기서는 네 가지 기본 도구(덧셈, 뺄셈, 곱셈, 나눗셈)를 구현하는 Lambda와 투자 관리(거래, 신용 조사,
정량 분석, 포트폴리오 관리)를 위해 생성된 75개 도구 정의 집합을 사용하여 두 번째 대상을 추가합니다. 투자 관리 도구 정의는
실제로 Lambda 함수에 구현되어 있지 않습니다. 많은 도구를 시연하기 위한 용도로만 추가합니다.

In [ ]:
calc_api_spec = read_apispec("./calc/calc-api.json")
print(f"API spec for calc has {len(calc_api_spec)} functions\n")
calc_lambda_arn = calc_lambda_resp["lambda_function_arn"]
print(f"Calc Lambda ARN: {calc_lambda_arn}")

time.sleep(5)
calcTargetId = create_gatewaytarget(
    gateway_id=gatewayId,
    lambda_arn=calc_lambda_arn,
    target_name="CalcTools",
    target_descr="Calculation Tools",
    api_spec=calc_api_spec,
)
print(f"CalcTools Target created with id: {calcTargetId} on gateway: {gatewayId}")

Gateway 검색의 강력한 기능을 시연하기 위해 이제 Calculator 대상의 복사본을 몇 개 더 추가하여
300개가 넘는 MCP 도구가 노출되도록 하겠습니다.

In [ ]:
def add_more_tools(gatewayId):
    time.sleep(10)
    calcTargetId = create_gatewaytarget(
        gateway_id=gatewayId,
        lambda_arn=calc_lambda_arn,
        target_name="Calc2",
        target_descr="Calculation 2 Tools",
        api_spec=calc_api_spec,
    )
    print(f"Calc2 Target created with id: {calcTargetId} on gateway: {gatewayId}")
    time.sleep(10)
    calcTargetId = create_gatewaytarget(
        gateway_id=gatewayId,
        lambda_arn=calc_lambda_arn,
        target_name="Calc3",
        target_descr="Calculation 3 Tools",
        api_spec=calc_api_spec,
    )
    print(f"Calc3 Target created with id: {calcTargetId} on gateway: {gatewayId}")
    time.sleep(10)
    calcTargetId = create_gatewaytarget(
        gateway_id=gatewayId,
        lambda_arn=calc_lambda_arn,
        target_name="Calc4",
        target_descr="Calculation 4 Tools",
        api_spec=calc_api_spec,
    )
    print(f"Calc4 Target created with id: {calcTargetId} on gateway: {gatewayId}")

In [ ]:
add_more_tools(gatewayId=gatewayId)

In [ ]:
resp = agentcore_client.list_gateway_targets(gatewayIdentifier=gatewayId)
targets = resp["items"]
for target in resp["items"]:
    print(f"{target['name']} - {target['description']}")

# Gateway에서 도구 검색

### 검색 전 MCP 도구 목록 조회 익히기

주어진 Gateway ID의 MCP 엔드포인트 URL을 가져오고 Gateway를 안전하게 사용하는 데 필요한
JWT OAuth 액세스 토큰을 가져오는 유틸리티 함수를 정의하겠습니다.

In [ ]:
def get_gateway_endpoint(gateway_id):
    response = agentcore_client.get_gateway(gatewayIdentifier=gateway_id)
    gateway_url = response["gatewayUrl"]
    return gateway_url

Gateway를 생성하고 대상을 추가했으므로 이제 해당 Gateway의 MCP URL을 가져오겠습니다.
Gateway ID를 사용하여 Gateway 컨트롤 플레인에서 엔드포인트 URL을 가져올 수 있습니다.

#### Gateway에서 MCP Inspector 사용

MCP 서버의 엔드포인트 URL과 JWT 전달자 토큰이 준비되었으므로 MCP Inspector 도구를 사용하여
MCP 서버를 살펴볼 수 있습니다. MCP Inspector는 모든 MCP 서버에 연결할 수 있는 오픈 소스 도구로,
제공되는 도구 목록을 조회하고 도구를 간편하게 호출할 수도 있습니다. 

터미널 창에서 `npx @modelcontextprotocol/inspector`를 입력하여 MCP Inspector를 실행합니다. 그런 다음
Gateway 엔드포인트 URL과 JWT 토큰을 붙여 넣어 연결합니다. 연결되면 List Tools와 Invoke Tool을 사용해 보세요.

다음은 샘플 스크린샷입니다.

![MCP Inspector 화면](images/mcp_inspector.png)

In [ ]:
gatewayEndpoint = get_gateway_endpoint(gateway_id=gatewayId)
print(f"Gateway Endpoint - MCP URL: {gatewayEndpoint}")

MCP 서버 보안은 OAuth를 기반으로 합니다. Gateway와 상호 작용하려면
IdP에서 JWT OAuth 액세스 토큰을 가져와야 합니다.

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
print(f"Bearer token: {jwtToken}")

In [ ]:
# !npx @modelcontextprotocol/inspector

#### jsonrpc를 사용하여 MCP 도구를 호출하거나 목록을 조회하는 헬퍼 함수 생성
jsonrpc를 사용하여 Gateway를 포함한 MCP Server가 노출하는 모든 MCP 도구를 호출하는
`invoke_gateway_tool` 헬퍼 함수를 정의하겠습니다. 엔드포인트 URL과 JWT 토큰을 지정하면
Gateway Target을 Gateway에 추가할 때 AgentCore Gateway가 제공한 모든 MCP 도구를
이 유틸리티로 호출할 수 있습니다.

In [ ]:
def invoke_gateway_tool(gateway_endpoint, jwt_token, tool_params):
    # print(f"Invoking tool {tool_params['name']}")

    requestBody = {
        "jsonrpc": "2.0",
        "id": 2,
        "method": "tools/call",
        "params": tool_params,
    }
    response = requests.post(
        gateway_endpoint,
        json=requestBody,
        headers={
            "Authorization": f"Bearer {jwt_token}",
            "Content-Type": "application/json",
        },
    )

    return response.json()

다음은 MCP의 `tools/list` 메서드를 사용하여 Gateway에서 사용할 수 있는 MCP 도구 목록을
조회하는 또 다른 유틸리티 함수입니다. Gateway ID와 JWT 토큰을 지정하면 해당 Gateway의
전체 도구 집합을 가져와 에이전트에서 바로 사용할 수 있는 형식의 목록으로 반환합니다. 반환된 목록에는
Agent에 전달하기 적합한 Strands Agents MCPAgentTool 객체가 포함됩니다. 

`tools/list` 호출은 페이지가 매겨지므로 `nextCursor` 필드에 더 이상 값이 없을 때까지
함수가 반복하면서 한 번에 한 페이지씩 도구를 가져와야 합니다. 이 유틸리티 함수는 HTTPS와
jsonrpc 프로토콜을 사용하여 엔드포인트를 직접 호출합니다. 이는 Strands Agents에서 제공하는
`MCPClient` 클래스보다 낮은 수준에서 도구 목록을 조회하는 방법입니다. 이 방법은 뒤에서 살펴보겠습니다.

In [ ]:
def get_all_agent_tools_from_mcp_endpoint(gateway_endpoint, jwt_token, client):
    more_tools = True
    tools_count = 0
    tools_list = []

    requestBody = {"jsonrpc": "2.0", "id": 2, "method": "tools/list", "params": {}}
    next_cursor = ""

    while more_tools:
        if tools_count == 0:
            requestBody["params"] = {}
        else:
            print("\nGetting next page of tools since a next cursor was returned\n")
            requestBody["params"] = {"cursor": next_cursor}

        headers = {
            "Authorization": f"Bearer {jwt_token}",
            "Content-Type": "application/json",
        }

        print(f"\n\nListing tools for gateway {gateway_endpoint}")

        response = requests.post(gateway_endpoint, json=requestBody, headers=headers)

        tools_json = response.json()
        tools_count += len(tools_json["result"]["tools"])

        for tool in tools_json["result"]["tools"]:
            mcp_tool = MCPTool(
                name=tool["name"],
                description=tool["description"],
                inputSchema=tool["inputSchema"],
            )
            mcp_agent_tool = MCPAgentTool(mcp_tool, client)
            short_descr = tool["description"][0:40] + "..."
            print(f"adding tool '{mcp_agent_tool.tool_name}' - {short_descr}")
            tools_list.append(mcp_agent_tool)

        if "nextCursor" in tools_json["result"]:
            next_cursor = tools_json["result"]["nextCursor"]
            more_tools = True
        else:
            more_tools = False

    print(f"\nTotal tools found: {tools_count}\n")
    return tools_list

이 헬퍼 함수를 사용하여 결과를 확인해 보겠습니다.

In [ ]:
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    all_tools = get_all_agent_tools_from_mcp_endpoint(
        gateway_endpoint=gatewayEndpoint, jwt_token=jwtToken, client=client
    )
    print(f"\nFound {len(all_tools)} tools using jsonrpc to list MCP tools\n")

#### 페이지 매김과 함께 Strands Agents list_tools_sync() 사용
Python 기반 MCP 클라이언트를 작성해 본 적이 있다면 클라이언트가 연결된 MCP Server에서
사용 가능한 도구 집합을 반환하는 `list_tools_sync()` 메서드가 익숙할 것입니다.
하지만 MCP 도구 목록에도 페이지가 매겨진다는 사실을 알고 계셨나요? 기본적으로는 반환되는 도구 중
첫 번째 일부만 가져옵니다. 단순한 MCP 서버에서는 이를 알아차리지 못할 수 있지만, 실제 환경의 많은
MCP 서버에서는 남은 페이지가 없을 때까지 코드를 반복하여 한 번에 한 페이지씩
도구를 가져와야 합니다. 다음 `get_all_mcp_tools_from_mcp_client` 유틸리티가 바로 이 작업을 수행합니다. 
이 함수는 주어진 Strands Agent MCP Client에서 전체 도구 목록을 반환합니다.

In [ ]:
def get_all_mcp_tools_from_mcp_client(client):
    more_tools = True
    tools = []
    pagination_token = None
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)
        tools.extend(tmp_tools)
        if tmp_tools.pagination_token is None:
            more_tools = False
        else:
            more_tools = True
            pagination_token = tmp_tools.pagination_token
    return tools

Gateway에서 이 함수를 사용하여 Python 클라이언트가 몇 개의 도구를 찾는지 확인해 보겠습니다. 
먼저 엔드포인트 URL과 JWT 전달자 토큰을 기반으로 MCPClient 객체를 생성합니다. 그런 다음
MCP 서버가 여러 페이지로 반환하는 전체 도구 집합을 가져옵니다.
앞서 추가한 대상을 고려하면 300개가 넘는 도구가 반환되어야 합니다.

In [ ]:
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")

지금까지 Gateway를 MCP Server로 사용하여 전체 도구 집합을 가져오는
세 가지 방법을 살펴보았습니다.

1. jsonrpc 직접 사용
2. Strands Agent MCPClient의 `list_tools_sync()` 메서드 사용
3. 내부적으로 jsonrpc를 사용하는 MCP Inspector 도구 사용

일반적으로 에이전트를 구축하는 개발자는 두 번째 방법을 사용합니다.

### Gateway 기본 제공 시맨틱 검색 도구 사용
이제 MCP 도구 목록에 추가되는 별도의 MCP 도구인 기본 제공 검색 도구를 사용하여
Gateway에서 첫 번째 시맨틱 검색을 수행해 보겠습니다.

먼저 MCP를 사용하여 검색 도구를 실행하는 간단한 유틸리티 함수를 정의하겠습니다.
도구 목록을 조회할 때와 마찬가지로 Gateway 엔드포인트와 JWT 토큰이 필요합니다. 그 외에는
검색 쿼리만 전달하면 됩니다. 나머지는 Gateway 검색 도구가 처리하며,
사용자를 대신하여 자동 관리하는 서버리스 벡터 스토어에서 쿼리와 일치하는 항목을 찾습니다.

In [ ]:
def tool_search(gateway_endpoint, jwt_token, query):
    toolParams = {
        "name": "x_amz_bedrock_agentcore_search",
        "arguments": {"query": query},
    }
    toolResp = invoke_gateway_tool(gateway_endpoint=gateway_endpoint, jwt_token=jwt_token, tool_params=toolParams)
    tools = toolResp["result"]["structuredContent"]["tools"]
    return tools

In [ ]:
start_time = time.time()
tools_found = tool_search(
    gateway_endpoint=gatewayEndpoint,
    jwt_token=jwtToken,
    query="find me 3 credit research tools",
)
end_time = time.time()
print(f"tool search via direct Gateway invocation took {(end_time - start_time):.2f} seconds")
print(f"Top tool: {tools_found[0]['name']}")

대부분의 경우 검색 결과가 1초 이내에 매우 빠르게 반환되는 것을 확인할 수 있습니다. 결과는
쿼리와 도구 메타데이터의 일치도를 기준으로 검색 관련성이 높은 순서대로 반환됩니다.
가장 관련성이 높은 도구가 목록의 맨 앞에 표시됩니다. 초기 검색 구현은 최대 10개의 결과를
반환합니다. 에이전트에서 이 도구를 모두 사용하거나, 관련성이 가장 높은 일부 도구만
선택할 수 있습니다.

# 도구가 많은 MCP 서버에서 Strands Agents 사용

먼저 Strands Agent에서 사용할 모델을 선택합니다. 
이 Notebook에서는 Amazon Bedrock 모델을 사용하지만 Strands와 AgentCore는
모든 LLM과 함께 사용할 수 있습니다.

In [ ]:
bedrockmodel = BedrockModel(
    model_id="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    temperature=0.7,
    streaming=True,
    boto_session=session,
)

#### AgentCore Gateway를 에이전트 도구로 사용하는 간단한 Strands Agent
이제 Strands Agent를 사용하여 AgentCore Gateway에서 제공하는 MCP Server를
얼마나 쉽게 활용할 수 있는지 살펴보겠습니다. 이 간단한 예제에서는
에이전트에 몇 개의 숫자를 더하도록 요청합니다.

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")

    simple_agent = Agent(model=bedrockmodel, tools=all_tools, callback_handler=null_callback_handler)
    result = simple_agent("add 100 plus 50 pass ")
    print(f"{result.message['content'][0]['text']}")

Strands Agents 프레임워크에서는 에이전트 이벤트 루프를 거치지 않고 MCP 도구를 직접 호출할 수도 있습니다.
Gateway 도구는 기본 MCP 도구로 노출되므로 Gateway 도구에도 같은 방식을 사용할 수 있습니다. 여기서는
`agent.tool.<tool_name>(args)` 구문을 사용하여 Gateway MCP 도구를 호출합니다.

```python
direct_result = simple_agent.tool.Calc2___add_numbers(firstNumber=10, secondNumber=20)
resp_json = json.loads(direct_result['content'][0]['text'])
```

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")

    simple_agent = Agent(model=bedrockmodel, tools=all_tools, callback_handler=null_callback_handler)
    direct_result = simple_agent.tool.Calc2___add_numbers(firstNumber=10, secondNumber=20)
    print(f"direct result = {direct_result}")

In [ ]:
def get_search_tool(client):
    mcp_tool = MCPTool(
        name="x_amz_bedrock_agentcore_search",
        description="A special tool that returns a trimmed down list of tools given a context. Use this tool only when there are many tools available and you want to get a subset that matches the provided context.",
        inputSchema={
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "search query to use for finding tools",
                }
            },
            "required": ["query"],
        },
    )
    return MCPAgentTool(mcp_tool, client)

In [ ]:
def search_using_strands(client, query):
    simple_agent = Agent(
        model=bedrockmodel,
        tools=[get_search_tool(client)],
        callback_handler=null_callback_handler,
    )

    direct_result = simple_agent.tool.x_amz_bedrock_agentcore_search(query=query)

    resp_json = json.loads(direct_result["content"][0]["text"])
    search_results = resp_json["tools"]
    # print(json.dumps(search_results, indent=4))
    return search_results

In [ ]:
def find_strands_tools(client, query, top_n):
    strands_mcp_tools = []
    results = search_using_strands(client, query)
    for tool in results[:top_n]:
        mcp_tool = MCPTool(
            name=tool["name"],
            description=tool["description"],
            inputSchema=tool["inputSchema"],
        )
        strands_mcp_tools.append(MCPAgentTool(mcp_tool, client))
    return strands_mcp_tools

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    simple_agent = Agent(
        model=bedrockmodel,
        tools=[get_search_tool(client)],
        callback_handler=null_callback_handler,
    )

    direct_result = simple_agent.tool.x_amz_bedrock_agentcore_search(query="find equity trading tools")

    resp_json = json.loads(direct_result["content"][0]["text"])
    search_results = resp_json["tools"]
    print(json.dumps(search_results, indent=4))

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    results = search_using_strands(client, "find trading tools")
    print(json.dumps(search_results[0], indent=4))

    results = search_using_strands(client, "find credit research tools")
    print(json.dumps(search_results[0], indent=4))

# Strands Agent에 도구 검색 결과 추가

이제 검색에서 반환된 도구를 Strands Agent에 추가하는 방법을 살펴보겠습니다.
코드를 간소화하기 위해 도구 검색 결과를 Strands MCPAgentTool 객체에
매핑하는 유틸리티 함수를 만들겠습니다. 검색 결과를 전달하고 그중 몇 개를
에이전트에 전달할지 지정하기만 하면 됩니다.

In [ ]:
def tools_to_strands_mcp_tools(tools, top_n):
    strands_mcp_tools = []
    for tool in tools[:top_n]:
        mcp_tool = MCPTool(
            name=tool["name"],
            description=tool["description"],
            inputSchema=tool["inputSchema"],
        )
        strands_mcp_tools.append(MCPAgentTool(mcp_tool, client))
    return strands_mcp_tools

In [ ]:
jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    agent = Agent(
        model=bedrockmodel,
        tools=find_strands_tools(
            client,
            "tools for doing addition, subtraction, multiplication, division",
            10,
        ),
    )
    result = agent("(10*2)/(5-3)")
    print(f"{result.message['content'][0]['text']}")

In [ ]:
%%time

jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))
with client:
    print("Searching for an ADDING tool from endpoint with full set of tools...")
    tools_found = tool_search(
        gateway_endpoint=gatewayEndpoint,
        jwt_token=jwtToken,
        query="tools for multiplying two numbers",
    )
    print(f"Top tool found: {tools_found[0]['name']}\n")

    agent = Agent(model=bedrockmodel, tools=tools_to_strands_mcp_tools(tools_found, 1))
    result = agent("10 * 70")
    print(f"{result.message['content'][0]['text']}")

지연 시간이 개선된 것을 확인해 보세요. Gateway 검색에서 찾은 일부 도구를 사용하는 이 예제는
수백 개의 도구에 의존하는 에이전트 호출보다 훨씬 빠릅니다.

# 도구 검색을 통한 3배의 지연 시간 개선 확인

이제 Strands Agent에서 Gateway MCP 도구를 사용하는 방법과 도구를 검색하여 에이전트에
추가하는 방법을 알게 되었으므로 검색의 강력한 기능을 살펴보겠습니다. 검색을 통해 얻을 수 있는
큰 폭의 지연 시간 감소와 입력 토큰 사용량 감소를 중점적으로 확인합니다.

지연 시간과 토큰 사용량 감소를 시연하기 위해 두 가지 접근 방식을 나란히 비교합니다.

1. **검색 미사용**. MCP 서버가 노출하는 전체 MCP 도구 집합(이 예제에서는 300개 이상)을 에이전트에 추가하고, 에이전트가 도구를 선택하여 호출하도록 합니다.
2. **검색 사용**. 두 번째 접근 방식에서는 현재 주제를 기반으로 검색하고 가장 관련성이 높은 도구만 에이전트에 전달합니다. 이를 입증하기 위해 서로 다른 도구 집합이 필요한 두 가지 주제인 수학(숫자 더하기)과 음식(레스토랑 예약)을 사용합니다.

지연 시간 분포를 정규화하고 의미 있는 비교 결과를 얻기 위해 각 접근 방식을 여러 번 반복합니다.
또한 개선 효과를 과장하지 않도록 검색 접근 방식에서는 에이전트 호출의 지연 시간뿐만 아니라
도구 검색 수행에 걸리는 지연 시간도 포함합니다. 각 반복에서 에이전트에 다음 두 가지 작업을
제공합니다.

1. 수학 작업 - 숫자 2개 더하기 
2. 음식 작업 - 레스토랑 예약

아래 결과는 지연 시간이 3배 줄고 입력 토큰 사용량은 그보다 더 크게 감소하는 이점을 보여 줍니다.
토큰 사용량 절감은 비용 절감으로 이어지지만, 입력 토큰 비용이 비교적 낮기 때문에 그 영향이 크지 않을 수 있습니다.
많은 모델 공급자에서 입력 토큰은 훨씬 저렴합니다. 그렇더라도 대규모 에이전트 배포에서는
입력 토큰 사용 비용도 누적될 수 있으므로 동적 검색은 에이전트 런타임 비용을
줄이는 데도 도움이 됩니다.

#### 전체 MCP 도구 집합을 사용하는 에이전트의 지연 시간 및 토큰 사용량 측정

In [ ]:
iterations = 2
full_tokens = light_tokens = 0
full_elapsed_time = light_elapsed_time = 0

jwtToken = utils.get_bearer_token(
    client_id=cognito_response["client_id"],
    username="testuser",
    password="MyPassword123!",
)
client = MCPClient(lambda: streamablehttp_client(f"{gatewayEndpoint}", headers={"Authorization": f"Bearer {jwtToken}"}))

In [ ]:
with client:
    all_tools = get_all_mcp_tools_from_mcp_client(client)
    print(f"\nFound {len(all_tools)} tools from list_tools_sync() on mcp client\n")
    heavy_agent = Agent(model=bedrockmodel, tools=all_tools, callback_handler=null_callback_handler)

    math_input = "add 100 plus <iteration>"
    food_input = "book me a table for 2 at Burger King under name Jo Smith at 7pm August <day>"

    print("using agent with ALL tools...")
    start_time = time.time()

    for i in range(iterations):
        result = heavy_agent(math_input.replace("<iteration>", str(i + 1)))
        print(f"{i + 1}) {result.message['content'][0]['text']}")

        result = heavy_agent(food_input.replace("<day>", str(i + 1)))
        print(f"{i + 1}) {result.message['content'][0]['text']}")

    end_time = time.time()
    full_tokens = result.metrics.accumulated_usage["totalTokens"]
    full_elapsed_time = end_time - start_time
    print(f"\nTotal time: {full_elapsed_time:.1f} s, tokens: {full_tokens:,d}\n")

#### Gateway 검색을 사용하는 에이전트의 지연 시간 및 토큰 사용량 측정
이제 검색을 호출하여 관련 도구를 찾은 다음 해당 도구만으로 에이전트를 호출하는
동적 접근 방식을 사용하겠습니다. 각 대화 턴마다 에이전트를 재설정하므로
이전 턴의 대화 기록으로 메시지 목록도 초기화한다는 점에 유의하세요.

In [ ]:
with client:
    print("using agent with ONLY tools from focused search...")
    start_time = time.time()
    messages = []

    light_agent = Agent()

    for i in range(iterations):
        print("Searching for an ADDING tool from endpoint with full set of tools...")
        tools_found = tool_search(
            gateway_endpoint=gatewayEndpoint,
            jwt_token=jwtToken,
            query="tools for simply adding two numbers",
        )
        print(f"Top tool found: {tools_found[0]['name']}\n")
        light_agent = Agent(
            model=bedrockmodel,
            tools=tools_to_strands_mcp_tools(tools_found, 1),
            messages=messages,
            callback_handler=null_callback_handler,
        )
        light_result = light_agent(math_input.replace("<iteration>", str(i + 1)))
        print(f"{i + 1}) {light_result.message['content'][0]['text']}")
        messages = light_agent.messages

        print("Searching for a RESTAURANT BOOKING tool from endpoint with full set of tools...")
        tools_found = tool_search(
            gateway_endpoint=gatewayEndpoint,
            jwt_token=jwtToken,
            query="tools for booking a restaurant reservation",
        )
        print(f"Top tool found: {tools_found[0]['name']}\n")
        light_agent = Agent(
            model=bedrockmodel,
            tools=tools_to_strands_mcp_tools(tools_found, 1),
            messages=messages,
            callback_handler=null_callback_handler,
        )
        light_result = light_agent(food_input.replace("<day>", str(i + 1)))
        print(f"{i + 1}) {light_result.message['content'][0]['text']}")
        messages = light_agent.messages
        light_tokens = light_result.metrics.accumulated_usage["totalTokens"]
    end_time = time.time()

    light_elapsed_time = end_time - start_time
    print(f"\nTotal time: {light_elapsed_time:.1f} s, tokens: {light_tokens:,d}\n")

#### 결과를 비교하여 검색의 이점 확인

In [ ]:
print(f"\n\nLatency without search: {full_elapsed_time:.1f}s, using search: {light_elapsed_time:.1f}s")
print(f"Tokens without search: {full_tokens:,d}, using search: {light_tokens:,d}")

# 마무리
이 튜토리얼에서는 Amazon Bedrock AgentCore Gateway와 기본 제공되는 완전 관리형
시맨틱 검색 기능을 학습했습니다. 다음 내용을 살펴보았습니다.

- 시맨틱 검색이 활성화된 Gateway를 생성하는 방법
- 여러 Gateway Target을 추가하여 단일 엔드포인트에서 300개가 넘는 MCP 도구를 노출하는 방법
- 세 가지 접근 방식으로 Gateway의 도구 목록을 조회하는 방법
- 기본 제공 시맨틱 검색 도구를 사용하여 관련 도구를 찾는 방법
- 검색을 Strands Agent와 통합하는 방법
- 수백 개의 도구가 있는 서버를 사용하는 에이전트와 시맨틱 검색으로 특정 주제에 맞게 도구를 좁히는 에이전트의 성능을 비교하는 방법

AgentCore Gateway 검색은 더 고급 사용 사례에도 유용합니다. 검색을 컨트롤 플레인 API뿐만 아니라
기본 MCP 도구로 제공하면 에이전트가 새로운 MCP 서버를 더 자율적으로 탐색하고 런타임에
새로운 기능을 찾아 더 어려운 문제를 해결할 수 있습니다.
또한 검색은 MCP 레지스트리와 에이전트 개발자가 새로운 에이전트를 설계하고 구축하도록
지원하는 데 중요한 기반입니다.

# 리소스 정리

먼저 AgentCore Gateway 리소스를 정리하는 헬퍼 함수를 정의하겠습니다.

In [ ]:
def delete_gatewaytarget(gateway_id):
    response = agentcore_client.list_gateway_targets(gatewayIdentifier=gateway_id)

    print(f"Found {len(response['items'])} targets for the gateway")

    for target in response["items"]:
        print(f"Deleting target with Name: {target['name']} and Id: {target['targetId']}")

        response = agentcore_client.delete_gateway_target(gatewayIdentifier=gateway_id, targetId=target["targetId"])
        time.sleep(20)


def delete_gateway(gateway_id):
    agentcore_client.delete_gateway(gatewayIdentifier=gateway_id)

### Gateway Target 삭제

In [ ]:
delete_gatewaytarget(gateway_id=gatewayId)

### Gateway 자체 삭제

In [ ]:
delete_gateway(gateway_id=gatewayId)

In [ ]:
lambda_arns = [
    calc_lambda_resp["lambda_function_arn"],
    restaurant_lambda_resp["lambda_function_arn"],
]

for arn in lambda_arns:
    if utils.delete_gateway_lambda(arn):
        print(f"Deleted Lambda: {arn}")
    else:
        print(f"Lambda {arn} not found or deletion failed")

In [ ]:
# Gateway 역할 정리
if utils.delete_gateway_iam_role():
    print("Gateway IAM role deleted")
else:
    print("Gateway IAM role not found or deletion failed")

# Cognito 정리
if utils.delete_cognito_user_pool():
    print("Cognito pool deleted")
else:
    print("✗ Failed to delete Cognito pool")